# Vision-Language Model (VLM)
LLaVA-style: "Visual Instruction Tuning" (Liu et al., 2023)

**Idea:** encode the image into patch tokens with a ViT, map them into the language model's embedding space with a small projector, and let a **decoder-only** language model read `[image tokens; text tokens]` and write the caption one token at a time.

Where this comes from:
- **CLIP** (Radford et al., 2021): ViT image encoder + text encoder trained contrastively - gives the vision tower most VLMs start from. It *matches* images and text, it can't *generate* text.
- **Flamingo** (Alayrac et al., 2022): frozen LM + **cross-attention** layers inserted between LM blocks to look at image features.
- **LLaVA** (Liu et al., 2023): simplest recipe - **no cross-attention**, just project image tokens and **prepend** them to the text. That's what we build here (tiny, from scratch, trained end to end).

### VLM vs original Transformer (Vaswani et al., 2017) vs ViT
| | Transformer (NLP) | ViT | VLM (this notebook) |
|---|---|---|---|
| Input | source + target word ids | image patches | image patches **and** caption characters |
| Parts used | encoder + decoder (with cross-attention) | encoder only | ViT encoder -> linear projector -> **decoder-only** LM (no cross-attention) |
| How the two sides talk | decoder cross-attends to encoder output | - | image tokens are **concatenated in front** of the text; plain self-attention sees both |
| Positional encoding | fixed sin/cos | learned | learned (separate for image and text) |
| Norm placement | Post-LN | Pre-LN | Pre-LN |
| Attention mask | causal in decoder | none | none in ViT; **causal** in the LM |
| Output | next-token distribution per position | one class from `[CLS]` | next-character distribution per text position |
| Loss | cross-entropy on target tokens | cross-entropy on class | cross-entropy on **text tokens only** (no loss on image positions) |

Shape flow: `(N, 1, 28, 28) -> patches (N, 49, Dv) -> ViT (N, 49, Dv) -> projector (N, 49, D)`, `text (N, T) -> embed (N, T, D)`, `concat (N, 49+T, D) -> causal decoder x L -> logits on text positions (N, T, V)`

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
random.seed(42)

## 1. Data - MNIST + generated captions
MNIST has labels, not captions, so we **write captions from the label**: a random template per sample, plus sometimes an extra fact (odd / even).

- Several templates for the same image means the text is partly unpredictable from the image - the model has to learn *which parts* depend on the image (the digit) and which are just "language" (the template).
- The extra fact ("which is odd") is a small reasoning step: the model must connect the digit it sees to a property never shown as pixels.

In [ ]:
WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
TEMPLATES = [
    "a handwritten digit {w}",
    "the number {d}",
    "this is {a} {w}",
    "a picture of {a} {w}",
    "the digit {d}, written by hand",
]

def make_caption(label, template=None, fact=None):
    w = WORDS[label]
    a = "an" if w[0] in "aeiou" else "a"
    template = random.choice(TEMPLATES) if template is None else template
    fact = random.random() < 0.5 if fact is None else fact
    text = template.format(w=w, d=label, a=a)
    if fact:
        text += ", which is " + ("even" if label % 2 == 0 else "odd")
    return text

for _ in range(5):
    y = random.randrange(10)
    print(y, "->", make_caption(y))

### Character-level tokenizer
Vocabulary = every character that can appear in a caption + 3 special tokens:
- `<pad>` (0): fills short captions to a fixed length, ignored by the loss
- `<bos>` (1): "start writing" - the first text token the LM sees
- `<eos>` (2): "stop" - the model learns when the caption ends

In [ ]:
all_captions = [make_caption(y, t, f) for y in range(10) for t in TEMPLATES for f in (False, True)]
chars = sorted(set("".join(all_captions)))
itos = ["<pad>", "<bos>", "<eos>"] + chars
stoi = {c: i for i, c in enumerate(itos)}
PAD, BOS, EOS = 0, 1, 2
vocab_size = len(itos)
max_len = max(len(c) for c in all_captions) + 2  # + <bos> + <eos>

def encode(text):
    # "the number 7" -> [BOS, t, h, e, ..., 7, EOS, PAD, ...]  (max_len,)
    ids = [BOS] + [stoi[c] for c in text] + [EOS]
    return torch.tensor(ids + [PAD] * (max_len - len(ids)))

def decode(ids):
    out = []
    for i in ids:
        i = int(i)
        if i == EOS:
            break
        if i > EOS:
            out.append(itos[i])
    return "".join(out)

print("vocab size:", vocab_size, "| max_len:", max_len)
print("".join(chars))
print(encode("the number 7")[:16], "->", decode(encode("the number 7")))

In [ ]:
class CaptionedMNIST(Dataset):
    # returns image (1, 28, 28) and caption ids (max_len,); a new random caption every time it's read
    def __init__(self, train):
        tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
        self.mnist = datasets.MNIST("../data", train=train, download=True, transform=tf)

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, i):
        img, label = self.mnist[i]
        return img, encode(make_caption(label)), label

train_ds = CaptionedMNIST(train=True)
test_ds = CaptionedMNIST(train=False)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

img, ids, label = train_ds[0]
print(img.shape, ids.shape, label, "->", decode(ids))

## 2. Multi-head self-attention (with optional causal mask)
Same scaled dot-product attention as the Transformer (full formula below).

- Vision encoder: `causal=False` - every patch sees every patch (like ViT).
- Language model: `causal=True` - position $i$ may only look at positions $\le i$, so it can't peek at the character it has to predict. Future positions get $-\infty$ before the softmax -> weight 0.

#### Formula: Scaled dot-product attention with causal mask
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V, \qquad Q = XW_Q,\ K = XW_K,\ V = XW_V$$
$$M_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases} \quad \text{(decoder)}, \qquad M = 0 \quad \text{(vision encoder)}$$
- $X \in \mathbb{R}^{L \times D}$: input tokens ($L = 49$ in the ViT, $L = 49 + T$ in the decoder)
- $W_Q, W_K, W_V \in \mathbb{R}^{D \times d_k}$: learned projections (fused in `self.qkv`)
- $Q, K, V$: queries, keys, values, each $L \times d_k$
- $d_k$: per-head dim ($D/h = 128/4 = 32$)
- $QK^T \in \mathbb{R}^{L \times L}$: similarity of every token pair
- $\sqrt{d_k}$: keeps scores at variance ~1 so softmax doesn't saturate
- $M$: causal mask; $e^{-\infty} = 0$, so token $i$ gets zero weight on every later token $j > i$ (`masked_fill(~tril, -inf)`)
- text token $i$ therefore sees all 49 image tokens and text tokens $\le i$

#### Formula: Multi-head attention
$$\text{MHA}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\,W_O, \qquad \text{head}_i = \text{Attention}(XW_Q^i, XW_K^i, XW_V^i)$$
- $h$: number of heads (4, in both ViT and LM)
- $W_Q^i, W_K^i, W_V^i$: projections of head $i$
- Concat: stacks head outputs back to $L \times h d_k = L \times D$
- $W_O \in \mathbb{R}^{D \times D}$: output projection (`self.proj`)

#### Formula: Softmax
$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}$$
- $x \in \mathbb{R}^n$: vector of scores (attention row, $n = L$; or LM logits, $n = V$)
- output: non-negative, sums to 1 (a probability distribution)

In [ ]:
class MultiHeadAttention(nn.Module):
    # (N, T, D) -> (N, T, D)
    def __init__(self, dim, num_heads, dropout=0.0, causal=False):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.causal = causal
        self.qkv = nn.Linear(dim, 3 * dim)   # Q, K, V in one matmul
        self.proj = nn.Linear(dim, dim)      # output projection W_O
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        N, T, D = x.shape
        qkv = self.qkv(x).reshape(N, T, 3, self.num_heads, self.head_dim)  # (N, T, 3D) -> (N, T, 3, heads, head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # each (N, heads, T, head_dim)

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5  # (N, heads, T, T)
        if self.causal:
            mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()  # (T, T), lower triangle = allowed
            att = att.masked_fill(~mask, float("-inf"))  # (T, T) broadcast over (N, heads)
        att = att.softmax(dim=-1)
        att = self.dropout(att)

        out = att @ v                        # (N, heads, T, head_dim)
        out = out.transpose(1, 2).reshape(N, T, D)  # concat heads -> (N, T, D)
        return self.proj(out)                # (N, T, D)

x = torch.randn(1, 4, 8)
m = MultiHeadAttention(8, 2, causal=True)
print(m(x)[0, 0] - m(x[:, :1])[0, 0])  # position 0 ignores the future -> zeros

## 3. Transformer block (Pre-LN)
Identical to the ViT block (Pre-LN residual formula below).

The **decoder block** is the same block with `causal=True`. Unlike the original Transformer decoder there is **no cross-attention** sub-layer (GPT-style): the image reaches the text through ordinary self-attention because the image tokens sit in the same sequence.

#### Formula: LayerNorm
$$\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta, \qquad \mu = \frac{1}{D}\sum_{i=1}^{D} x_i,\quad \sigma^2 = \frac{1}{D}\sum_{i=1}^{D}(x_i - \mu)^2$$
- $x \in \mathbb{R}^D$: one token's feature vector ($D = 128$), normalized per token
- $\mu, \sigma^2$: mean and variance over the $D$ features
- $\epsilon$: small constant for numerical stability ($10^{-5}$)
- $\gamma, \beta \in \mathbb{R}^D$: learned scale and shift
- $\odot$: element-wise product

#### Formula: Pre-LN block (encoder and decoder)
$$z'_\ell = z_{\ell-1} + \text{MHA}(\text{LN}(z_{\ell-1})), \qquad z_\ell = z'_\ell + \text{MLP}(\text{LN}(z'_\ell))$$
$$\text{logits}_t = \text{LN}(z_{L}^{P+t})\,W_{lm} + b_{lm}$$
- $z_{\ell-1}$: input to block $\ell$; $z'_\ell$: after attention sub-layer; $z_\ell$: block output
- MLP: $\text{Linear}(D \to 4D) \to \text{GELU} \to \text{Linear}(4D \to D)$, i.e. $128 \to 512 \to 128$
- $L$: number of blocks (4 in the ViT, 4 in the LM); ViT blocks use $M = 0$, LM blocks use the causal $M$
- $z_L^{P+t}$: final decoder output at text position $t$ (offset by the $P = 49$ image tokens)
- $W_{lm} \in \mathbb{R}^{D \times V}$, $b_{lm}$: `lm_head`, turns a hidden vector into next-character scores

#### Formula: GELU
$$\text{GELU}(x) = x\,\Phi(x) \approx 0.5\,x\left(1 + \tanh\!\left[\sqrt{2/\pi}\,\left(x + 0.044715\,x^3\right)\right]\right)$$
- $x$: scalar input (applied element-wise)
- $\Phi(x) = P(X \le x),\ X \sim \mathcal{N}(0,1)$: standard normal CDF
- tanh form: cheap approximation (`nn.GELU()` default uses the exact erf form)

In [ ]:
class Block(nn.Module):
    # (N, T, D) -> (N, T, D)
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.0, causal=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads, dropout, causal)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_ratio * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_ratio * dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # Pre-LN, (N, T, D)
        x = x + self.mlp(self.ln2(x))   # (N, T, D) -> (N, T, 4D) -> (N, T, D)
        return x

## 4. Vision encoder (small ViT)
Same as `vit.ipynb`, but:
- 1 input channel, 28x28 image, patch 4 -> 7x7 = **49 patch tokens**
- **no `[CLS]` token and no classifier head** - we keep *all* patch outputs, because the LM should be able to look at any part of the image (LLaVA also uses the patch tokens, not just CLS).

#### Formula: Patch embedding (vision encoder input)
$$z_0 = [\,x_p^1 E;\ x_p^2 E;\ \dots;\ x_p^{P} E\,] + E_{pos}^{img}, \qquad P = \frac{HW}{p^2}$$
- $x \in \mathbb{R}^{H \times W \times C}$: input image (here $28 \times 28 \times 1$)
- $p$: patch size (4); $C$: channels (1); $D_v$: vision embedding dim (128)
- $P$: number of patches / image tokens ($28 \cdot 28 / 4^2 = 49$)
- $x_p^i \in \mathbb{R}^{p^2 C}$: $i$-th patch, flattened (16 values)
- $E \in \mathbb{R}^{(p^2 C) \times D_v}$: shared linear patch projection (the `Conv2d`)
- $E_{pos}^{img} \in \mathbb{R}^{P \times D_v}$: learned positional embedding (no `[CLS]` token here)
- $[\,\cdot\,;\,\cdot\,]$: concatenation along the sequence axis; $z_0 \in \mathbb{R}^{49 \times 128}$

In [ ]:
class PatchEmbedding(nn.Module):
    # (N, C, H, W) -> (N, num_patches, D)
    # e.g. (N, 1, 28, 28), patch 4 -> 7x7 = 49 patches -> (N, 49, D)
    def __init__(self, img_size, patch_size, in_ch, dim):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)                 # (N, D, 7, 7)
        x = x.flatten(2).transpose(1, 2) # (N, 49, D)
        return x


class VisionEncoder(nn.Module):
    # (N, 1, 28, 28) -> (N, 49, Dv)
    def __init__(self, img_size=28, patch_size=4, in_ch=1, dim=128, depth=4, num_heads=4, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_ch, dim)
        self.num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, dim))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(dim, num_heads, dropout=dropout) for _ in range(depth)])
        self.ln = nn.LayerNorm(dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        x = self.patch_embed(x)               # (N, 49, Dv)
        x = self.dropout(x + self.pos_embed)  # + (1, 49, Dv) broadcast over N
        x = self.blocks(x)                    # (N, 49, Dv)
        return self.ln(x)                     # (N, 49, Dv)

print(VisionEncoder()(torch.randn(2, 1, 28, 28)).shape)

## 5. Projector
Image features and word embeddings live in different spaces (different sizes, different meaning of each dimension). The projector is a learned map **vision space -> text embedding space**, so an image patch becomes a "visual word" the LM can read.

- LLaVA-1: a single `nn.Linear` (used here)
- LLaVA-1.5: 2-layer MLP (`Linear -> GELU -> Linear`)
- Flamingo instead uses a Perceiver Resampler (compresses to a fixed number of tokens) + cross-attention

#### Formula: Projector
$$H_{img} = Z_v W_p + b_p$$
- $Z_v \in \mathbb{R}^{P \times D_v}$: ViT output (after final LN), $49 \times 128$
- $W_p \in \mathbb{R}^{D_v \times D}$, $b_p \in \mathbb{R}^{D}$: learned projection (`Projector.proj`), $128 \to 128$
- $D$: language model width (128)
- $H_{img} \in \mathbb{R}^{P \times D}$: image tokens in the text embedding space

In [ ]:
class Projector(nn.Module):
    # (N, 49, Dv) -> (N, 49, D)
    def __init__(self, vision_dim, text_dim):
        super().__init__()
        self.proj = nn.Linear(vision_dim, text_dim)

    def forward(self, x):
        return self.proj(x)  # (N, 49, Dv) -> (N, 49, D)

## 6. Full VLM
1. image -> ViT -> projector -> **image tokens** `(N, 49, D)`
2. caption ids -> `nn.Embedding` + learned text positions -> **text tokens** `(N, T, D)`
3. `concat` -> `(N, 49 + T, D)` -> causal decoder blocks -> LN -> `lm_head`
4. keep only the logits at **text positions** -> `(N, T, V)`

With the causal mask over the whole sequence, text position $t$ sees all 49 image tokens (they come before it) and text tokens $\le t$. Image positions produce outputs too, but we throw them away - there's nothing to predict there.

Training input is `ids[:, :-1]` and the target is `ids[:, 1:]` (shift by one): logits at `<bos>` predict the 1st character, and so on.

#### Formula: Text embedding and the joint sequence
$$H_{txt} = [\,e(w_1);\ \dots;\ e(w_T)\,] + E_{pos}^{txt}, \qquad X = [\,H_{img};\ H_{txt}\,] \in \mathbb{R}^{(P+T) \times D}$$
- $w_1 = \texttt{<bos>}, w_2, \dots$: caption token ids (characters); $T \le$ `max_len` $- 1$
- $e(\cdot)$: embedding lookup, a row of `tok_embed` $\in \mathbb{R}^{V \times D}$
- $V$: vocabulary size (characters in the captions + 3 special tokens; printed in section 1)
- $E_{pos}^{txt} \in \mathbb{R}^{\text{max\_len} \times D}$: learned text positional embedding
- $X$: input to the decoder, 49 image tokens followed by $T$ text tokens

In [ ]:
class VLM(nn.Module):
    # images (N, 1, 28, 28), ids (N, T) -> logits (N, T, V)
    def __init__(self, vocab_size, max_len, vision_dim=128, vision_depth=4,
                 dim=128, depth=4, num_heads=4, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.vision = VisionEncoder(dim=vision_dim, depth=vision_depth, num_heads=num_heads, dropout=dropout)
        self.projector = Projector(vision_dim, dim)

        self.tok_embed = nn.Embedding(vocab_size, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, dim))  # text positions
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.Sequential(*[Block(dim, num_heads, mlp_ratio, dropout, causal=True) for _ in range(depth)])
        self.ln = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, images, ids):
        img = self.projector(self.vision(images))           # (N, 49, D)
        T = ids.shape[1]
        txt = self.tok_embed(ids) + self.pos_embed[:, :T]   # (N, T, D)

        x = torch.cat([img, txt], dim=1)                    # (N, 49+T, D)
        x = self.blocks(self.dropout(x))                    # (N, 49+T, D)
        x = self.ln(x[:, img.shape[1]:])                    # text positions only -> (N, T, D)
        return self.lm_head(x)                              # (N, T, V)

model = VLM(vocab_size, max_len).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"  vision {sum(p.numel() for p in model.vision.parameters()) / 1e6:.2f}M"
      f" | projector {sum(p.numel() for p in model.projector.parameters()) / 1e6:.3f}M")
print(model(torch.randn(2, 1, 28, 28, device=device), torch.zeros(2, 10, dtype=torch.long, device=device)).shape)

## 7. Training
Plain **next-token prediction**, end to end (vision encoder, projector and LM all learn together):
- loss = cross-entropy between logits at text position $t$ and the character at $t+1$
- `ignore_index=PAD`: padding positions don't count
- no loss on image positions (they were already dropped in `forward`)

Same optimizer recipe as ViT: **AdamW** + linear warmup + cosine decay, plus gradient clipping (standard for language models).

Real LLaVA trains in two stages instead: (1) freeze a pre-trained CLIP ViT and a pre-trained LLM, train only the projector on image-caption pairs; (2) unfreeze the LM and fine-tune on instruction data. We have no pre-trained parts, so everything trains at once.

#### Formula: Autoregressive LM loss (cross-entropy on text tokens)
$$p_\theta(w_{2:T+1} \mid I) = \prod_{t=1}^{T} p_\theta(w_{t+1} \mid I,\ w_{\le t}), \qquad \mathcal{L} = -\frac{1}{\sum_t m_t}\sum_{t=1}^{T} m_t \log p_\theta(w_{t+1} \mid I,\ w_{\le t})$$
- $I$: the image (enters only through the 49 image tokens)
- $w_{\le t}$: caption characters up to position $t$ (input `ids[:, :-1]`); $w_{t+1}$: target (`ids[:, 1:]`)
- $p_\theta(w_{t+1} \mid \cdot) = \text{softmax}(\text{logits}_t)_{w_{t+1}}$: predicted probability of the true next character
- $m_t = \mathbb{1}[w_{t+1} \ne \texttt{<pad>}]$: mask, padding targets are ignored (`ignore_index=PAD`)
- no terms for image positions - the model is never asked to predict pixels
- averaged over all non-pad text tokens in the batch (batch size 128); $\theta$: all parameters (ViT + projector + LM)

#### Formula: Gradient clipping (by global norm)
$$g \leftarrow g \cdot \min\!\left(1,\ \frac{c}{\lVert g \rVert_2}\right)$$
- $g$: gradient of all parameters concatenated; $\lVert g \rVert_2$: its L2 norm
- $c$: max norm (1.0); rescales only when the norm is larger, direction unchanged

#### Formula: AdamW
$$g_t = \nabla_\theta \mathcal{L}(\theta_{t-1})$$
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat v_t = \frac{v_t}{1-\beta_2^t}$$
$$\theta_t = \theta_{t-1} - \eta_t\left(\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon} + \lambda\,\theta_{t-1}\right)$$
- $\theta_t$: parameters at step $t$; $g_t$: (clipped) gradient
- $m_t$: EMA of gradients (1st moment); $v_t$: EMA of squared gradients (2nd moment)
- $\beta_1, \beta_2$: decay rates (defaults 0.9, 0.999)
- $\hat m_t, \hat v_t$: bias-corrected moments
- $\eta_t$: learning rate (peak $10^{-3}$, set by the schedule below)
- $\epsilon$: stability constant ($10^{-8}$)
- $\lambda$: weight decay (0.05), applied directly to $\theta$ (decoupled from the gradient)

#### Formula: LR schedule: linear warmup + cosine decay
$$\eta_t = \eta \cdot \frac{t}{T_w} \quad (t \le T_w), \qquad \eta_t = \eta_{min} + \tfrac{1}{2}(\eta - \eta_{min})\left(1 + \cos\frac{\pi t'}{T}\right) \quad (t > T_w)$$
- $\eta$: peak LR ($10^{-3}$); $\eta_{min}$: final LR (0, PyTorch default)
- $t$: current optimizer step (scheduler steps per batch; 469 batches/epoch)
- $T_w$: warmup steps (1 epoch = 469 steps); code starts at $0.01\,\eta$ (`start_factor=0.01`) rather than 0
- $t' = t - T_w$: steps since warmup ended
- $T$: cosine length (9 epochs $= 4221$ steps)

In [ ]:
epochs = 10
lr = 1e-3
warmup_epochs = 1

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_epochs * len(train_loader))
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(epochs - warmup_epochs) * len(train_loader))
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epochs * len(train_loader)])

def lm_loss(model, images, ids):
    logits = model(images, ids[:, :-1])                # (N, T-1, V)
    targets = ids[:, 1:]                               # (N, T-1) next character
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=PAD)  # (N*(T-1), V) vs (N*(T-1),) -> scalar

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0
    for images, ids, _ in loader:
        total += lm_loss(model, images.to(device), ids.to(device)).item()
    return total / len(loader)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, ids, _ in train_loader:
        images, ids = images.to(device), ids.to(device)  # (N, 1, 28, 28), (N, T)
        loss = lm_loss(model, images, ids)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()  # per step, not per epoch
        total_loss += loss.item()

    print(f"epoch {epoch+1:2d} | train loss {total_loss / len(train_loader):.4f} | test loss {evaluate(model, test_loader):.4f}")

## 8. Generation (greedy)
Autoregressive decoding: start from `[<bos>]` (optionally + a text **prompt**), run the model, take the most likely next character (`argmax`), append it, repeat until `<eos>` or `max_len`.

No KV-cache here: every step re-runs the whole sequence (simple, and fine for ~50 characters). All images in a batch are decoded together.

Greedy decoding with no prompt always picks the single most likely template. Giving a prompt such as `"the number "` forces a template, and the model only has to fill in what depends on the image.

#### Formula: Greedy decoding
$$\hat w_{t+1} = \arg\max_{v \in \{1..V\}}\ p_\theta(v \mid I,\ \hat w_{\le t}), \qquad \hat w_1 = \texttt{<bos>}\ (+\ \text{prompt})$$
- $\hat w_{t}$: generated characters, fed back in as input
- stops at `<eos>` or after `max_len` tokens

In [ ]:
@torch.no_grad()
def generate(model, images, prompt=""):
    # images (N, 1, 28, 28) -> list of N strings
    model.eval()
    N = images.shape[0]
    ids = torch.tensor([BOS] + [stoi[c] for c in prompt], device=device).repeat(N, 1)  # (N, 1+len(prompt))
    done = torch.zeros(N, dtype=torch.bool, device=device)
    while ids.shape[1] < max_len and not done.all():
        logits = model(images, ids)[:, -1]              # (N, V) last position only
        next_id = logits.argmax(-1)                     # greedy -> (N,)
        next_id = next_id.masked_fill(done, PAD)        # finished captions just get padding
        ids = torch.cat([ids, next_id[:, None]], dim=1)  # (N, t) -> (N, t+1)
        done |= next_id == EOS
    return [prompt + decode(row[1 + len(prompt):]) for row in ids]

idx = random.sample(range(len(test_ds)), 8)
images = torch.stack([test_ds[i][0] for i in idx]).to(device)
true_caps = [decode(test_ds[i][1]) for i in idx]
gen_caps = generate(model, images)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, gen, true in zip(axes.flat, images.cpu(), gen_caps, true_caps):
    ax.imshow(img[0], cmap="gray")
    ax.set_title(f"gen:  {gen}\ntrue: {true}", fontsize=8, loc="left")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Prompting
Same images, different prompts. The model continues the text in a way that matches both the prompt and the picture.

In [ ]:
prompts = ["", "the number ", "this is ", "a picture of ", "the digit "]
for i in range(4):
    print(f"image {idx[i]} (label {test_ds.mnist.targets[idx[i]].item()})")
    for p in prompts:
        print(f"   prompt {p!r:18} -> {generate(model, images[i:i+1], p)[0]}")

### Does it actually look at the image?
Prompt every test image with `"the number "` and check whether the next generated character is the right digit. This turns captioning into a classification accuracy we can compare with `vit.ipynb` / a CNN.

In [ ]:
correct = total = 0
for images, _, labels in test_loader:
    caps = generate(model, images.to(device), "the number ")
    correct += sum(c[len("the number ")] == str(y.item()) for c, y in zip(caps, labels))
    total += len(labels)
print(f"digit accuracy from captions: {correct / total:.4f}")

## Notes
- The test loss can't go to 0: the template and the "which is odd/even" suffix are chosen at random, so part of every caption is not predictable from the image. The loss that *is* reducible is on the digit characters and the odd/even word - those are what the image determines.
- Digit accuracy via the prompt `"the number "` should end up close to a normal MNIST classifier (~98-99%), even though the model was never trained with a classification loss - the label only ever appears as text.
- Mask choice: here the causal mask covers the **whole** sequence (LLaVA does this too). A **prefix-LM** mask (image tokens attend to each other in both directions, text stays causal) is a common alternative; it matters less here because the ViT already mixed the patches bidirectionally.
- Scaling this up = the real recipe: swap the ViT for a pre-trained CLIP ViT-L/14 (576 patch tokens), the tiny LM for a pre-trained LLM (Vicuna/LLaMA, 7B+), the char tokenizer for BPE, and MNIST captions for image-text pairs + instruction data.
- Cost: sequence length is `49 + T`, so image tokens make up most of the attention cost ($O((P+T)^2)$). This is why later VLMs compress/resample image tokens (Flamingo's Perceiver Resampler, Q-Former in BLIP-2).